# AIC2026 — Pipeline từ Dataset mới đến FAISS

Luồng của notebook:

1. Nạp `aic2026_keyframe_new/final_dataset`.
2. Đưa keyframe và metadata vào `/kaggle/working/aic_practice` bằng symlink/copy nhẹ.
3. Nạp code mới nhất từ model `update-script...`.
4. Build `clip_row_mapping.jsonl`.
5. Encode toàn bộ CLIP features, có hỗ trợ resume.
6. Build và kiểm tra `clip.index`.
7. Đóng gói FAISS + mapping thành Dataset để dùng cho notebook Search.

Notebook không build Object/OCR/ASR.

In [1]:
from pathlib import Path
import os

INPUT_ROOT = Path("/kaggle/input")
PROJECT_ROOT = Path("/kaggle/working/aic_practice")
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
DATA_DIR = PROJECT_ROOT / "data"
KEYFRAME_TARGET = DATA_DIR / "keyframes"
METADATA_TARGET = DATA_DIR / "metadata"
FEATURE_DIR = PROJECT_ROOT / "features" / "clip"

for directory in (
    PROJECT_ROOT,
    SCRIPTS_DIR,
    ARTIFACTS_DIR,
    DATA_DIR,
    KEYFRAME_TARGET,
    METADATA_TARGET,
    FEATURE_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

mounted_roots = []
for category in ("datasets", "models", "notebooks"):
    category_root = INPUT_ROOT / category
    if not category_root.exists():
        continue
    for owner_root in category_root.iterdir():
        if not owner_root.is_dir():
            continue
        for mounted_root in owner_root.iterdir():
            if mounted_root.is_dir():
                mounted_roots.append(mounted_root)

print("CÁC INPUT ĐÃ GẮN:")
for root in mounted_roots:
    print("-", root)

CÁC INPUT ĐÃ GẮN:
- /kaggle/input/datasets/ltnngoc/aic2026-keyframe-new
- /kaggle/input/models/blonton/update-script-v2


In [2]:
def normalize_name(value):
    return str(value).casefold().replace("_", "-").replace(" ", "-")


def find_mount(*keywords, required=True):
    values = [normalize_name(keyword) for keyword in keywords]
    matches = [
        root
        for root in mounted_roots
        if all(value in normalize_name(root) for value in values)
    ]
    if not matches:
        if required:
            raise FileNotFoundError(f"Không tìm thấy Kaggle Input: {keywords}")
        return None
    if len(matches) > 1:
        print(f"Nhiều Input khớp {keywords}:")
        for match in matches:
            print(" -", match)
    return matches[0]


KEYFRAME_DATASET_ROOT = find_mount("aic2026", "keyframe", "new")
CODE_DATASET_ROOT = find_mount("update", "script")
FINAL_DATASET_ROOT = KEYFRAME_DATASET_ROOT / "final_dataset"
KEYFRAME_SOURCE = FINAL_DATASET_ROOT / "keyframes"

assert FINAL_DATASET_ROOT.is_dir(), f"Thiếu {FINAL_DATASET_ROOT}"
assert KEYFRAME_SOURCE.is_dir(), f"Thiếu {KEYFRAME_SOURCE}"

print("Dataset:", KEYFRAME_DATASET_ROOT)
print("Final dataset:", FINAL_DATASET_ROOT)
print("Keyframes:", KEYFRAME_SOURCE)
print("Code:", CODE_DATASET_ROOT)

Dataset: /kaggle/input/datasets/ltnngoc/aic2026-keyframe-new
Final dataset: /kaggle/input/datasets/ltnngoc/aic2026-keyframe-new/final_dataset
Keyframes: /kaggle/input/datasets/ltnngoc/aic2026-keyframe-new/final_dataset/keyframes
Code: /kaggle/input/models/blonton/update-script-v2


In [3]:
import shutil


def find_file(root, filename, max_depth=14):
    root = Path(root)
    root_depth = len(root.parts)
    for current, directories, files in os.walk(root):
        current_path = Path(current)
        depth = len(current_path.parts) - root_depth
        if depth >= max_depth:
            directories[:] = []
        if filename in files:
            return current_path / filename
    return None


BUILD_MAPPING_SOURCE = find_file(CODE_DATASET_ROOT, "build_mapping.py")
assert BUILD_MAPPING_SOURCE, "Model code thiếu build_mapping.py"
CODE_SOURCE_DIR = BUILD_MAPPING_SOURCE.parent

required_scripts = (
    "build_mapping.py",
    "encode_clip_features.py",
    "build_faiss_index.py",
    "search_types.py",
    "stdio_setup.py",
)

for filename in required_scripts:
    assert (CODE_SOURCE_DIR / filename).is_file(), f"Code thiếu {filename}"

copied = []
for source in CODE_SOURCE_DIR.glob("*.py"):
    shutil.copy2(source, SCRIPTS_DIR / source.name)
    copied.append(source.name)

print("✅ Nguồn code:", CODE_SOURCE_DIR)
print("✅ Đã copy:", len(copied), "script")

✅ Nguồn code: /kaggle/input/models/blonton/update-script-v2/pytorch/default/1
✅ Đã copy: 33 script


## Gắn keyframe và metadata

Chỉ liệt kê các thư mục video; không quét từng ảnh và không copy ảnh vào Working.

In [4]:
video_sources = {
    entry.name: Path(entry.path)
    for entry in os.scandir(KEYFRAME_SOURCE)
    if entry.is_dir()
}

assert video_sources, f"Không có video trong {KEYFRAME_SOURCE}"

created = 0
existing = 0
for video_id, source in sorted(video_sources.items()):
    target = KEYFRAME_TARGET / video_id
    if target.is_symlink():
        if target.resolve() == source.resolve():
            existing += 1
            continue
        raise RuntimeError(f"Symlink đang trỏ sai: {target} -> {target.resolve()}")
    if target.exists():
        raise RuntimeError(f"Target đã tồn tại và không phải symlink: {target}")
    target.symlink_to(source, target_is_directory=True)
    created += 1

print("Video nguồn:", len(video_sources))
print("Symlink mới:", created)
print("Đã tồn tại đúng:", existing)

Video nguồn: 873
Symlink mới: 873
Đã tồn tại đúng: 0


In [5]:
metadata_names = (
    "dataset-metadata.json",
    "dataset_manifest.json",
    "keyframes.jsonl",
    "temporal_relations.jsonl",
    "validation_report.json",
    "videos.jsonl",
)

for filename in metadata_names:
    source = FINAL_DATASET_ROOT / filename
    if source.is_file():
        shutil.copy2(source, METADATA_TARGET / filename)
        print("✅", filename)
    else:
        print("⚠️ Thiếu tùy chọn:", filename)

KEYFRAMES_METADATA = METADATA_TARGET / "keyframes.jsonl"
MANIFEST_PATH = METADATA_TARGET / "dataset_manifest.json"

assert KEYFRAMES_METADATA.is_file(), "Thiếu keyframes.jsonl"
assert MANIFEST_PATH.is_file(), "Thiếu dataset_manifest.json"

✅ dataset-metadata.json
✅ dataset_manifest.json
✅ keyframes.jsonl
✅ temporal_relations.jsonl
✅ validation_report.json
✅ videos.jsonl


In [6]:
import json

manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8-sig"))
expected_videos = int(manifest["video_count"])
expected_keyframes = int(manifest["keyframe_count"])

with KEYFRAMES_METADATA.open("r", encoding="utf-8-sig") as file:
    first_metadata_row = json.loads(next(line for line in file if line.strip()))

required_fields = {
    "video_id",
    "frame_index",
    "timestamp_ms",
    "keyframe_path",
}
missing_fields = required_fields - set(first_metadata_row)

print("Manifest version:", manifest.get("dataset_version"))
print("Video mong đợi:", expected_videos)
print("Keyframe mong đợi:", expected_keyframes)
print("Video đã gắn:", len(video_sources))
print("Metadata mẫu:", first_metadata_row)

assert len(video_sources) == expected_videos
assert not missing_fields, f"Metadata thiếu trường: {missing_fields}"
print("✅ Dataset mới hợp lệ")

Manifest version: aic_dataset_v1
Video mong đợi: 873
Keyframe mong đợi: 196590
Video đã gắn: 873
Metadata mẫu: {'keyframe_id': 'L21_V001_K000000', 'video_id': 'L21_V001', 'frame_index': 0, 'timestamp_ms': 0, 'keyframe_path': 'keyframes/L21_V001/000000.jpg', 'shot_id': 'L21_V001_S001'}
✅ Dataset mới hợp lệ


## Build mapping mới

In [7]:
from pathlib import Path
import shutil

PROJECT_ROOT = Path(
    "/kaggle/working/aic_practice"
)

METADATA_DIR = (
    PROJECT_ROOT / "data/metadata"
)

METADATA_AUX_DIR = (
    PROJECT_ROOT / "data/metadata_aux"
)

METADATA_AUX_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# Các JSONL này không phải metadata từng keyframe
auxiliary_jsonl = [
    "videos.jsonl",
    "temporal_relations.jsonl",
]

for filename in auxiliary_jsonl:
    source = METADATA_DIR / filename
    destination = (
        METADATA_AUX_DIR / filename
    )

    if source.is_file():
        if destination.exists():
            destination.unlink()

        shutil.move(
            str(source),
            str(destination),
        )

        print(
            "✅ Đã chuyển:",
            source,
            "→",
            destination,
        )
    else:
        print(
            "Không nằm trong metadata:",
            filename,
        )

remaining_jsonl = sorted(
    path.name
    for path in METADATA_DIR.glob("*.jsonl")
)

print(
    "\nJSONL còn trong data/metadata:",
    remaining_jsonl,
)

assert remaining_jsonl == [
    "keyframes.jsonl"
], (
    "data/metadata chỉ được chứa keyframes.jsonl; "
    f"hiện có: {remaining_jsonl}"
)

print(
    "✅ Metadata đã sẵn sàng build mapping"
)

✅ Đã chuyển: /kaggle/working/aic_practice/data/metadata/videos.jsonl → /kaggle/working/aic_practice/data/metadata_aux/videos.jsonl
✅ Đã chuyển: /kaggle/working/aic_practice/data/metadata/temporal_relations.jsonl → /kaggle/working/aic_practice/data/metadata_aux/temporal_relations.jsonl

JSONL còn trong data/metadata: ['keyframes.jsonl']
✅ Metadata đã sẵn sàng build mapping


In [8]:
%cd /kaggle/working/aic_practice
!python -u scripts/build_mapping.py

/kaggle/working/aic_practice
Đã tạo mapping theo layout BTC
Số video: 873
Số keyframe: 196590
Mapping: /kaggle/working/aic_practice/artifacts/clip_row_mapping.jsonl
  - L21_V001: 500 keyframe
  - L21_V002: 451 keyframe
  - L21_V003: 526 keyframe
  - L21_V005: 362 keyframe
  - L21_V006: 427 keyframe
  - L21_V007: 351 keyframe
  - L21_V008: 489 keyframe
  - L21_V009: 476 keyframe
  - L21_V010: 481 keyframe
  - L21_V011: 479 keyframe
  - L21_V012: 369 keyframe
  - L21_V013: 501 keyframe
  - L21_V014: 502 keyframe
  - L21_V015: 532 keyframe
  - L21_V016: 428 keyframe
  - L21_V017: 369 keyframe
  - L21_V018: 490 keyframe
  - L21_V019: 437 keyframe
  - L21_V021: 389 keyframe
  - L21_V022: 382 keyframe
  - L21_V023: 508 keyframe
  - L21_V024: 504 keyframe
  - L21_V025: 494 keyframe
  - L21_V026: 489 keyframe
  - L21_V027: 532 keyframe
  - L21_V028: 376 keyframe
  - L21_V029: 507 keyframe
  - L21_V030: 438 keyframe
  - L21_V031: 449 keyframe
  - L22_V001: 547 keyframe
  - L22_V002: 445 keyfram

In [9]:
from pathlib import Path

MAPPING_PATH = Path(
    "/kaggle/working/aic_practice/"
    "artifacts/clip_row_mapping.jsonl"
)

assert MAPPING_PATH.is_file(), (
    "Build mapping vẫn chưa tạo output"
)

print("✅ Mapping:", MAPPING_PATH)
print(
    "Dung lượng:",
    round(
        MAPPING_PATH.stat().st_size
        / 1024**2,
        2,
    ),
    "MB",
)

✅ Mapping: /kaggle/working/aic_practice/artifacts/clip_row_mapping.jsonl
Dung lượng: 45.43 MB


## Encode CLIP features

Bật GPU trước khi chạy. Encoder ghi một file NPY cho mỗi video và hỗ trợ resume nếu session bị ngắt.

In [10]:
%pip install -q open_clip_torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.3 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [11]:
import torch

print("CUDA:", torch.cuda.is_available())
print(
    "GPU:",
    torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Không có",
)
assert torch.cuda.is_available(), "Hãy bật Accelerator GPU trước khi encode"

os.environ["AIC_CLIP_BATCH_SIZE"] = "128"
os.environ["AIC_CLIP_RESUME"] = "1"
os.environ["AIC_VALIDATE_KEYFRAMES"] = "sample"

print("✅ Resume bật; batch size=128")

CUDA: True
GPU: Tesla T4
✅ Resume bật; batch size=128


In [12]:
%cd /kaggle/working/aic_practice
!python -u scripts/encode_clip_features.py

/kaggle/working/aic_practice
Kiểm tra mẫu 1746 đường dẫn của 873 video
Thiết bị: cuda
Model: ViT-B-32-quickgelu
Checkpoint: openai
Số keyframe: 196590
Đang tải CLIP model...
open_clip_model.safetensors: 100%|████████████| 605M/605M [00:05<00:00, 105MB/s]
  - L21_V001: 500 vector → L21_V001.npy
  - L21_V002: 451 vector → L21_V002.npy
  - L21_V003: 526 vector → L21_V003.npy
  - L21_V005: 362 vector → L21_V005.npy
  - L21_V006: 427 vector → L21_V006.npy
  - L21_V007: 351 vector → L21_V007.npy
  - L21_V008: 489 vector → L21_V008.npy
  - L21_V009: 476 vector → L21_V009.npy
  - L21_V010: 481 vector → L21_V010.npy
  - L21_V011: 479 vector → L21_V011.npy
  - L21_V012: 369 vector → L21_V012.npy
  - L21_V013: 501 vector → L21_V013.npy
  - L21_V014: 502 vector → L21_V014.npy
  - L21_V015: 532 vector → L21_V015.npy
  - L21_V016: 428 vector → L21_V016.npy
  - L21_V017: 369 vector → L21_V017.npy
  - L21_V018: 490 vector → L21_V018.npy
  - L21_V019: 437 vector → L21_V019.npy
  - L21_V021: 389 vector 

In [13]:
import numpy as np

npy_files = sorted(FEATURE_DIR.glob("*.npy"))
valid_files = []
invalid_files = []
total_vectors = 0

for path in npy_files:
    try:
        array = np.load(path, mmap_mode="r")
        if array.ndim != 2 or array.shape[0] <= 0:
            invalid_files.append(path)
            continue
        valid_files.append(path)
        total_vectors += int(array.shape[0])
    except Exception:
        invalid_files.append(path)

print("NPY hợp lệ:", len(valid_files))
print("NPY lỗi:", len(invalid_files))
print("Tổng vector:", total_vectors)
print("Mong đợi:", expected_keyframes)

assert not invalid_files, f"NPY lỗi: {invalid_files[:10]}"
assert len(valid_files) == expected_videos
assert total_vectors == expected_keyframes
print("✅ CLIP features đầy đủ")

NPY hợp lệ: 873
NPY lỗi: 0
Tổng vector: 196590
Mong đợi: 196590
✅ CLIP features đầy đủ


## Build và kiểm tra FAISS

In [14]:
%pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 45.3 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [15]:
%cd /kaggle/working/aic_practice
!python -u scripts/build_faiss_index.py

/kaggle/working/aic_practice
Tạo FAISS index thành công
Index type: IndexFlatIP
Index dimension: 512
Index ntotal: 196590
Bất biến C1: index.ntotal (196590) = tổng keyframe = số dòng mapping (196590) ✓
Index path: /kaggle/working/aic_practice/artifacts/clip.index
Manifest: /kaggle/working/aic_practice/artifacts/clip_index_manifest.json


In [16]:
from pathlib import Path
from datetime import datetime
import json
import hashlib
import faiss

PROJECT_ROOT = Path(
    "/kaggle/working/aic_practice"
)

ARTIFACTS_DIR = (
    PROJECT_ROOT / "artifacts"
)

MAPPING_PATH = (
    ARTIFACTS_DIR
    / "clip_row_mapping.jsonl"
)

FAISS_PATH = (
    ARTIFACTS_DIR
    / "clip.index"
)

assert MAPPING_PATH.is_file(), (
    f"Thiếu mapping: {MAPPING_PATH}"
)

assert FAISS_PATH.is_file(), (
    f"Thiếu FAISS: {FAISS_PATH}"
)


# Đếm lại mapping, không phụ thuộc cell trước
mapping_count = 0
first_mapping_row = None
last_mapping_row = None

with MAPPING_PATH.open(
    "r",
    encoding="utf-8-sig",
) as file:
    for line in file:
        line = line.strip()

        if not line:
            continue

        row = json.loads(line)

        if first_mapping_row is None:
            first_mapping_row = row

        last_mapping_row = row
        mapping_count += 1


# Đọc FAISS
index = faiss.read_index(
    str(FAISS_PATH)
)

print("Loại index:", type(index).__name__)
print("Số vector FAISS:", f"{index.ntotal:,}")
print("Số dòng mapping:", f"{mapping_count:,}")
print("Số chiều:", index.d)
print(
    "Dung lượng:",
    round(
        FAISS_PATH.stat().st_size
        / 1024**3,
        3,
    ),
    "GB",
)

assert mapping_count > 0
assert first_mapping_row["vector_index"] == 0
assert (
    last_mapping_row["vector_index"]
    == mapping_count - 1
)

assert index.ntotal == mapping_count, (
    f"FAISS có {index.ntotal:,} vector "
    f"nhưng mapping có {mapping_count:,} dòng"
)

print("✅ FAISS khớp mapping")

Loại index: IndexFlatIP
Số vector FAISS: 196,590
Số dòng mapping: 196,590
Số chiều: 512
Dung lượng: 0.375 GB
✅ FAISS khớp mapping


In [17]:
def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open("rb") as file:
        while True:
            chunk = file.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


checkpoint = {
    "created_at": datetime.now().isoformat(),
    "dataset": (
        "aic2026_keyframe_new/"
        "final_dataset"
    ),
    "mapping_rows": mapping_count,
    "faiss_vectors": int(index.ntotal),
    "faiss_dimension": int(index.d),
    "mapping_sha256": sha256_file(
        MAPPING_PATH
    ),
    "mapping_path": (
        "artifacts/"
        "clip_row_mapping.jsonl"
    ),
    "faiss_path": (
        "artifacts/clip.index"
    ),
}

CHECKPOINT_PATH = (
    ARTIFACTS_DIR
    / "faiss_checkpoint.json"
)

CHECKPOINT_PATH.write_text(
    json.dumps(
        checkpoint,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    json.dumps(
        checkpoint,
        ensure_ascii=False,
        indent=2,
    )
)

print("✅ Checkpoint:", CHECKPOINT_PATH)

{
  "created_at": "2026-08-27T16:51:27.317425",
  "dataset": "aic2026_keyframe_new/final_dataset",
  "mapping_rows": 196590,
  "faiss_vectors": 196590,
  "faiss_dimension": 512,
  "mapping_sha256": "7c5910f828ef1402c254e7d4438f14aa18d58b367a752c6ee4634e63207300d8",
  "mapping_path": "artifacts/clip_row_mapping.jsonl",
  "faiss_path": "artifacts/clip.index"
}
✅ Checkpoint: /kaggle/working/aic_practice/artifacts/faiss_checkpoint.json


## Đóng gói Dataset FAISS cho notebook Search

In [18]:
import shutil
import zipfile

EXPORT_ROOT = Path("/kaggle/working/aic2026_faiss_ready")
EXPORT_ARTIFACTS = EXPORT_ROOT / "artifacts"
EXPORT_METADATA = EXPORT_ROOT / "data" / "metadata"
EXPORT_ARTIFACTS.mkdir(parents=True, exist_ok=True)
EXPORT_METADATA.mkdir(parents=True, exist_ok=True)

shutil.copy2(FAISS_PATH, EXPORT_ARTIFACTS / "clip.index")
shutil.copy2(MAPPING_PATH, EXPORT_ARTIFACTS / "clip_row_mapping.jsonl")
shutil.copy2(CHECKPOINT_PATH, EXPORT_ARTIFACTS / "faiss_checkpoint.json")
shutil.copy2(MANIFEST_PATH, EXPORT_METADATA / "dataset_manifest.json")

ZIP_PATH = Path("/kaggle/working/aic2026_faiss_ready.zip")
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(
    ZIP_PATH,
    mode="w",
    compression=zipfile.ZIP_STORED,
) as archive:
    for path in EXPORT_ROOT.rglob("*"):
        if path.is_file():
            archive.write(path, arcname=path.relative_to(EXPORT_ROOT))

print("✅ Dataset folder:", EXPORT_ROOT)
print("✅ ZIP:", ZIP_PATH)
print("Dung lượng ZIP:", round(ZIP_PATH.stat().st_size / 1024**3, 3), "GB")

✅ Dataset folder: /kaggle/working/aic2026_faiss_ready
✅ ZIP: /kaggle/working/aic2026_faiss_ready.zip
Dung lượng ZIP: 0.419 GB


In [19]:
from IPython.display import FileLink, display

display(
    FileLink(
        str(ZIP_PATH),
        result_html_prefix="Nhấn để tải FAISS + mapping: ",
    )
)

/kaggle/working/aic2026_faiss_ready.zip

In [20]:
from pathlib import Path
import shutil

PROJECT_ROOT = Path(
    "/kaggle/working/aic_practice"
)

FEATURE_SOURCE = (
    PROJECT_ROOT
    / "features"
    / "clip"
)

FEATURE_EXPORT = (
    Path("/kaggle/working")
    / "aic2026_clip_features"
    / "features"
    / "clip"
)

assert FEATURE_SOURCE.is_dir(), (
    f"Không tìm thấy CLIP features: {FEATURE_SOURCE}"
)

FEATURE_EXPORT.mkdir(
    parents=True,
    exist_ok=True,
)

feature_files = sorted(
    FEATURE_SOURCE.glob("*.npy")
)

assert feature_files, (
    "Không tìm thấy file NPY"
)

print("Nguồn:", FEATURE_SOURCE)
print("Đích:", FEATURE_EXPORT)
print("Số NPY:", len(feature_files))

copied = 0
skipped = 0

for index, source in enumerate(
    feature_files,
    start=1,
):
    destination = (
        FEATURE_EXPORT / source.name
    )

    if (
        destination.is_file()
        and destination.stat().st_size
        == source.stat().st_size
    ):
        skipped += 1
    else:
        shutil.copy2(
            source,
            destination,
        )
        copied += 1

    if index % 50 == 0:
        print(
            f"Đã xử lý "
            f"{index}/{len(feature_files)}"
        )

print("✅ Copy mới:", copied)
print("✅ Bỏ qua đã có:", skipped)
print("✅ Output:", FEATURE_EXPORT)

Nguồn: /kaggle/working/aic_practice/features/clip
Đích: /kaggle/working/aic2026_clip_features/features/clip
Số NPY: 873
Đã xử lý 50/873
Đã xử lý 100/873
Đã xử lý 150/873
Đã xử lý 200/873
Đã xử lý 250/873
Đã xử lý 300/873
Đã xử lý 350/873
Đã xử lý 400/873
Đã xử lý 450/873
Đã xử lý 500/873
Đã xử lý 550/873
Đã xử lý 600/873
Đã xử lý 650/873
Đã xử lý 700/873
Đã xử lý 750/873
Đã xử lý 800/873
Đã xử lý 850/873
✅ Copy mới: 873
✅ Bỏ qua đã có: 0
✅ Output: /kaggle/working/aic2026_clip_features/features/clip


In [21]:
CLIP_MANIFEST_SOURCE = (
    PROJECT_ROOT
    / "features"
    / "clip_features_manifest.json"
)

CLIP_EXPORT_ROOT = (
    Path("/kaggle/working")
    / "aic2026_clip_features"
)

if CLIP_MANIFEST_SOURCE.is_file():
    shutil.copy2(
        CLIP_MANIFEST_SOURCE,
        CLIP_EXPORT_ROOT
        / "clip_features_manifest.json",
    )

    print("✅ Đã copy CLIP manifest")
else:
    print(
        "⚠️ Không có clip_features_manifest.json"
    )

✅ Đã copy CLIP manifest


In [22]:
from pathlib import Path
import zipfile
import numpy as np
import json

PROJECT_ROOT = Path(
    "/kaggle/working/aic_practice"
)

FEATURE_DIR = (
    PROJECT_ROOT
    / "features"
    / "clip"
)

CLIP_MANIFEST = (
    PROJECT_ROOT
    / "features"
    / "clip_features_manifest.json"
)

MAPPING_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "clip_row_mapping.jsonl"
)

ZIP_PATH = Path(
    "/kaggle/working/"
    "aic2026_clip_features.zip"
)

assert FEATURE_DIR.is_dir(), (
    f"Thiếu CLIP features: {FEATURE_DIR}"
)

npy_files = sorted(
    FEATURE_DIR.glob("*.npy")
)

assert npy_files, "Không tìm thấy NPY"

print("Đang kiểm tra", len(npy_files), "file...")

Đang kiểm tra 873 file...


In [23]:
valid_files = []
invalid_files = []
total_vectors = 0
dimensions = set()

for path in npy_files:
    try:
        array = np.load(
            path,
            mmap_mode="r",
        )

        if (
            array.ndim != 2
            or array.shape[0] <= 0
        ):
            invalid_files.append(path)
            continue

        valid_files.append(path)
        total_vectors += int(
            array.shape[0]
        )
        dimensions.add(
            int(array.shape[1])
        )

    except Exception as error:
        invalid_files.append(
            (path, str(error))
        )

print("NPY hợp lệ:", len(valid_files))
print("NPY lỗi:", len(invalid_files))
print("Tổng vector:", total_vectors)
print("Dimension:", dimensions)

assert len(valid_files) == 873
assert total_vectors == 196590
assert dimensions == {512}
assert not invalid_files

print("✅ CLIP features hợp lệ")

NPY hợp lệ: 873
NPY lỗi: 0
Tổng vector: 196590
Dimension: {512}
✅ CLIP features hợp lệ


In [24]:
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

print("Đang tạo ZIP...", flush=True)

with zipfile.ZipFile(
    ZIP_PATH,
    mode="w",
    compression=zipfile.ZIP_STORED,
) as archive:
    for index, path in enumerate(
        valid_files,
        start=1,
    ):
        archive.write(
            path,
            arcname=(
                f"features/clip/{path.name}"
            ),
        )

        if index % 50 == 0:
            print(
                f"Đã đóng gói "
                f"{index}/{len(valid_files)}",
                flush=True,
            )

    if CLIP_MANIFEST.is_file():
        archive.write(
            CLIP_MANIFEST,
            arcname=(
                "clip_features_manifest.json"
            ),
        )

    if MAPPING_PATH.is_file():
        archive.write(
            MAPPING_PATH,
            arcname=(
                "artifacts/"
                "clip_row_mapping.jsonl"
            ),
        )

print("✅ ZIP:", ZIP_PATH)
print(
    "Dung lượng:",
    round(
        ZIP_PATH.stat().st_size
        / 1024**3,
        3,
    ),
    "GB",
)

Đang tạo ZIP...
Đã đóng gói 50/873
Đã đóng gói 100/873
Đã đóng gói 150/873
Đã đóng gói 200/873
Đã đóng gói 250/873
Đã đóng gói 300/873
Đã đóng gói 350/873
Đã đóng gói 400/873
Đã đóng gói 450/873
Đã đóng gói 500/873
Đã đóng gói 550/873
Đã đóng gói 600/873
Đã đóng gói 650/873
Đã đóng gói 700/873
Đã đóng gói 750/873
Đã đóng gói 800/873
Đã đóng gói 850/873
✅ ZIP: /kaggle/working/aic2026_clip_features.zip
Dung lượng: 0.42 GB
